In [0]:
dbutils.widgets.text("catalog_param", "")
dbutils.widgets.text("source_schema_param", "")
dbutils.widgets.text("target_schema_param", "")
dbutils.widgets.text("raw_schema_param", "")
# 2. Capturar los valores inyectados por el YAML
catalog_name = dbutils.widgets.get("catalog_param")
source_schema = dbutils.widgets.get("source_schema_param")
target_schema = dbutils.widgets.get("target_schema_param")
#raw_schema = dbutils.widgets.get("raw_schema_param")
print(catalog_name)

In [0]:
# Catálogo automatizado para las dimensiones de Real Estate
tablas_a_procesar_inmuebles = [
    {
        "source_object": "inmuebles", # Tu tabla en la capa silver
        "target_object": "DimInmuebles",
        "key_cols_list": ["Id_Inmueble"], # Llave natural de la propiedad
        "surrogate_key": "DimInmuebleKey",
        "cdc_col": "extraction_date" # O "Fecha", la que uses para deltas
    },
    {
        "source_object": "inmuebles",
        "target_object": "DimUbicaciones",
        "key_cols_list": ["Estado_Oficial", "Municipio_Oficial"], # Llave natural compuesta
        "surrogate_key": "DimUbicacionKey",
        "cdc_col": "extraction_date"
    }
]

In [0]:
from pyspark.sql.functions import col, current_timestamp, md5, concat_ws, lit
from delta.tables import DeltaTable
from concurrent.futures import ThreadPoolExecutor, as_completed
def procesar_dimension_inmuebles(config, catalog_name, source_schema, target_schema, backdated_refresh=""):
    """
    Ejecuta un pipeline SCD Tipo 1 completo, utilizando llaves subrogadas determinísticas (MD5)
    para operaciones distribuidas de alto rendimiento.
    """
    source_object = config["source_object"]
    target_object = config["target_object"]
    key_cols_list = config["key_cols_list"]
    surrogate_key = config["surrogate_key"]
    cdc_col = config["cdc_col"]
    
    print(f"🚀 Iniciando procesamiento para: {target_object}...")
    
    # 1. Calcular last_load (High-Water Mark)
    if len(backdated_refresh) == 0:
        if spark.catalog.tableExists(f"{catalog_name}.{target_schema}.{target_object}"):
            last_load = spark.sql(f"SELECT max({cdc_col}) FROM {catalog_name}.{target_schema}.{target_object}").collect()[0][0]
            # Manejo de tabla existente pero vacía
            if last_load is None: 
                last_load = "1900-01-01"
        else:
            last_load = "1900-01-01"
    else:
        last_load = backdated_refresh

    # 2. Extraer df_src (Solo deltas)
    df_src = spark.sql(f"SELECT * FROM {catalog_name}.{source_schema}.{source_object} WHERE {cdc_col} > '{last_load}'")
    
    # Para tablas como DimUbicaciones, necesitamos asegurar que el origen sea único por sus llaves
    df_src = df_src.dropDuplicates(key_cols_list)
    
    if df_src.count() == 0:
        return f"✅ Sin datos nuevos para {target_object}."

    # 3. Crear df_trg
    if spark.catalog.tableExists(f"{catalog_name}.{target_schema}.{target_object}"): 
        key_cols_string_incremental = ", ".join(key_cols_list)
        df_trg = spark.sql(f"""SELECT {key_cols_string_incremental}, {surrogate_key}, create_date, update_date 
                               FROM {catalog_name}.{target_schema}.{target_object}""")
    else:
        key_cols_string_init = [f"CAST(NULL AS STRING) AS {i}" for i in key_cols_list]
        key_cols_string_init = ", ".join(key_cols_string_init)
        df_trg = spark.sql(f"""SELECT {key_cols_string_init}, 
                               CAST('0' AS STRING) AS {surrogate_key}, 
                               CAST('1900-01-01 00:00:00' AS timestamp) AS create_date, 
                               CAST('1900-01-01 00:00:00' AS timestamp) AS update_date 
                               WHERE 1=0""")

    # 4. JOIN Nativo de PySpark
    condicion_join = col(f"src.{key_cols_list[0]}") == col(f"trg.{key_cols_list[0]}")
    for i in key_cols_list[1:]:
        condicion_join &= col(f"src.{i}") == col(f"trg.{i}")

    df_join = df_src.alias("src").join(
        df_trg.alias("trg"),
        on=condicion_join,
        how="left"
    ).select(
        "src.*",
        f"trg.{surrogate_key}",
        "trg.create_date",
        "trg.update_date"
    )

    # 5. Filtrar Viejos y Nuevos
    df_old = df_join.filter(col(f'{surrogate_key}').isNotNull())
    df_new = df_join.filter(col(f'{surrogate_key}').isNull())

    # =====================================================================
    # 🔥 EL ARREGLO: LLAVE SUBROGADA HASHEADA (DETERMINÍSTICA E IDEMPOTENTE)
    # =====================================================================
    df_old_enr = df_old.withColumn('update_date', current_timestamp())

    # Generamos la llave uniendo las columnas naturales con un separador y aplicando MD5
    df_new_enr = df_new \
        .withColumn(surrogate_key, md5(concat_ws("||", *[col(c) for c in key_cols_list]))) \
        .withColumn('create_date', current_timestamp()) \
        .withColumn('update_date', current_timestamp())    
    # =====================================================================

    # 7. Unir DataFrames
    df_union = df_old_enr.unionByName(df_new_enr)

    # 8. MERGE Final a Delta
    if spark.catalog.tableExists(f"{catalog_name}.{target_schema}.{target_object}"):
        dlt_obj = DeltaTable.forName(spark, f"{catalog_name}.{target_schema}.{target_object}")
        dlt_obj.alias("trg").merge(df_union.alias("src"), f"trg.{surrogate_key} = src.{surrogate_key}")\
                            .whenMatchedUpdateAll(condition = f"src.{cdc_col} >= trg.{cdc_col}")\
                            .whenNotMatchedInsertAll()\
                            .execute()
    else: 
        df_union.write.format("delta")\
                .mode("append")\
                .saveAsTable(f"{catalog_name}.{target_schema}.{target_object}")
    print(f"{catalog_name}.{target_schema}.{target_object}")            
    return f"✅ Completado con éxito: {target_object}!"


In [0]:
%python
hilos_paralelos = 4 

print(f"🔥 Iniciando carga simultánea de {len(tablas_a_procesar_inmuebles)} tablas con {hilos_paralelos} hilos...\n")

with ThreadPoolExecutor(max_workers=hilos_paralelos) as executor:
    # Enviamos todas las configuraciones a la función maestra al mismo tiempo
    futuros = {
        executor.submit(
            procesar_dimension_inmuebles, 
            config, 
            catalog_name, 
            source_schema, 
            target_schema
        ): config["target_object"] 
        for config in tablas_a_procesar_inmuebles
    }
    
    # Monitoreamos las respuestas conforme vayan terminando
    for futuro in as_completed(futuros):
        nombre_tabla = futuros[futuro]
        try:
            resultado = futuro.result() 
            print(resultado)
        except Exception as e:
            print(f"🔴 ERROR en la tabla {nombre_tabla}. Detalle: {str(e)}")

print("\n🎯 Procesamiento de la Capa Gold (Dimensiones) finalizado.")
print(catalog_name)